In [ ]:
!pip install transformers datasets peft accelerate sentence-transformers evaluate google-generativeai

In [ ]:
!pip install --upgrade transformers

In [ ]:
import torch
import random
import numpy as np
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

# unify format
def format_alpaca(x):
    return {"instruction": x["instruction"], "output": x["output"]}

def format_dolly(x):
    return {"instruction": x["instruction"], "output": x["response"]}

alpaca = alpaca.map(format_alpaca, remove_columns=alpaca.column_names)
dolly = dolly.map(format_dolly, remove_columns=dolly.column_names)

data = concatenate_datasets([alpaca, dolly])
print(len(data))

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    return text.strip()

data = data.map(lambda x: {
    "instruction": clean_text(x["instruction"]),
    "output": clean_text(x["output"])
})

data = data.filter(lambda x: len(x["instruction"]) > 5 and len(x["output"]) > 5)

In [ ]:
# First split: train (80%) and temp (20%)
split1 = data.train_test_split(test_size=0.2, seed=42)

train_data = split1["train"]
temp_data = split1["test"]

# Second split: val (10%) and test (10%)
split2 = temp_data.train_test_split(test_size=0.5, seed=42)

val_data = split2["train"]
test_data = split2["test"]

print(len(train_data), len(val_data), len(test_data))

In [ ]:
def char_reverse(text):
    return text[::-1]

def word_reverse(text):
    return " ".join(text.split()[::-1])

def create_irrelevant(data):
    outputs = [x["output"] for x in data]
    random.shuffle(outputs)
    return [{"instruction": d["instruction"], "output": outputs[i]} for i, d in enumerate(data)]

In [ ]:
#gemini couterfactual
import google.generativeai as genai

genai.configure(api_key="API KEY, INSERT FROM GOOGLE STUDIO")

model_gemini = genai.GenerativeModel("gemini-1.5-flash")

def generate_counterfactual(text):
    prompt = f"Give a wrong but plausible answer for: {text}"
    try:
        response = model_gemini.generate_content(prompt)
        return response.text
    except:
        return "Incorrect answer"

In [ ]:
import random

# --- Transformations ---

def char_reverse(text):
    return text[::-1]

def word_reverse(text):
    return " ".join(text.split()[::-1])


def apply_transformation(data, type_="char"):
    data = list(data)  # ensure list

    new_data = []

    for d in data:
        if type_ == "char":
            new_output = char_reverse(d["output"])
        elif type_ == "word":
            new_output = word_reverse(d["output"])
        else:
            new_output = d["output"]

        new_data.append({
            "instruction": d["instruction"],
            "output": new_output
        })

    return new_data


# --- Robust Mixing ---

def mix_data(clean, corrupted, ratio=0.5):
    clean = list(clean)
    corrupted = list(corrupted)

    # shuffle both (important)
    random.shuffle(clean)
    random.shuffle(corrupted)

    k = int(len(clean) * ratio)

    mixed = clean[:k] + corrupted[:k]

    # final shuffle so model doesn't see blocks
    random.shuffle(mixed)

    return mixed


# --- APPLY (Example: char reversal) ---

char_data = apply_transformation(train_data, "char")
train_mixed = mix_data(train_data, char_data, ratio=0.5)

print("Clean size:", len(train_data))
print("Mixed size:", len(train_mixed))

In [ ]:
#tokeniser + model list
from transformers import AutoTokenizer, AutoModelForCausalLM
def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens
model_names = [

    "sshleifer/tiny-gpt2",
    "distilgpt2",
    "facebook/opt-125m"
]


tokenizer = AutoTokenizer.from_pretrained(model_names[0])
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
 ''' existing
    "facebook/opt-350m",
    "EleutherAI/pythia-410m",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/phi-2",
    "Qwen/Qwen2-0.5B",

    # new small
    "facebook/opt-125m",
    "EleutherAI/pythia-160m",
    "distilgpt2",
    "gpt2",
    "sshleifer/tiny-gpt2",

    # medium
    "facebook/opt-1.3b",
    "EleutherAI/pythia-1b",
    "microsoft/phi-1_5",
    "tiiuae/falcon-rw-1b",
    "Qwen/Qwen2-1.5B"
]'''

In [ ]:
# tokenisation function (FIXED)

def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256   # reduced for stability
    )

    tokens["labels"] = tokens["input_ids"].copy()  # 🔴 CRITICAL

    return tokens


# --- APPLY TOKENIZATION ---
train_dataset = [tokenize(x) for x in train_mixed]
val_dataset = [tokenize(x) for x in val_data]

from datasets import Dataset

train_dataset = Dataset.from_list(train_dataset)
val_dataset = Dataset.from_list(val_dataset)

# convert to torch tensors
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

In [ ]:
from peft import LoraConfig, get_peft_model

def get_lora_model(model, model_name):

    # Try different possible modules safely
    possible_modules = [
        ["c_attn"],                 # GPT2, OPT
        ["q_proj", "v_proj"],       # LLaMA, Qwen
        ["k_proj", "out_proj"],     # fallback
    ]

    for modules in possible_modules:
        try:
            config = LoraConfig(
                r=4,
                lora_alpha=8,
                target_modules=modules,
                lora_dropout=0.05,
                bias="none"
            )
            return get_peft_model(model, config)
        except ValueError:
            continue

    print(f"LoRA not applied for {model_name}, using base model")
    return model

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM

results = []

for model_name in model_names:
    print("\nTraining:", model_name)

    safe_name = model_name.replace("/", "_")

    # Load model
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # Apply LoRA (fixed)
    model = get_lora_model(model, model_name)

    # Training arguments (CPU safe + fast)
    args = TrainingArguments(
        output_dir=f"./results/{safe_name}",
        per_device_train_batch_size=1,
        num_train_epochs=1,
        max_steps=50,
        logging_steps=10,
        save_strategy="no"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset
    )

    # Train
    trainer.train()

    # Save model reference
    results.append((model_name, model))

In [ ]:
# inference
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # PRINT OUTPUT
    print("\n--- INPUT ---")
    print(prompt)
    print("\n--- OUTPUT ---")
    print(text)
    print("\n---------------------------")

    return text

In [ ]:
generate(results[0][1], "Explain gravity")

In [ ]:
#evaluation (semantic similarity)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sim_model = SentenceTransformer("all-mpnet-base-v2")

def compute_similarity(pred, ref):
    emb1 = sim_model.encode([pred])
    emb2 = sim_model.encode([ref])
    return cosine_similarity(emb1, emb2)[0][0]



In [ ]:
!pip install rouge_score

In [ ]:
import evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load models once
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
sim_model = SentenceTransformer("all-mpnet-base-v2")

def compute_similarity(pred, ref):
    emb1 = sim_model.encode([pred])
    emb2 = sim_model.encode([ref])
    return cosine_similarity(emb1, emb2)[0][0]

def compute_metrics(pred, ref):
    bleu_score = bleu.compute(predictions=[pred], references=[[ref]])
    rouge_score = rouge.compute(predictions=[pred], references=[ref])

    return bleu_score["bleu"], rouge_score["rougeL"]

In [ ]:
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
final_results = []

for model_name, model in results:
    print(f"\nEvaluating: {model_name}")

    sim_scores = []
    bleu_scores = []
    rouge_scores = []

    for d in test_data[:100]:

        #HANDLE BOTH FORMATS SAFELY
        if isinstance(d, dict):
            instruction = d.get("instruction", "")
            reference = d.get("output", "")
        else:
            instruction = str(d)
            reference = str(d)

        # Generate prediction
        pred = generate(model, instruction)

        # Metrics
        sim = compute_similarity(pred, reference)
        bleu_s, rouge_s = compute_metrics(pred, reference)

        sim_scores.append(sim)
        bleu_scores.append(bleu_s)
        rouge_scores.append(rouge_s)

    final_results.append({
        "model": model_name,
        "similarity": np.mean(sim_scores),
        "bleu": np.mean(bleu_scores),
        "rouge": np.mean(rouge_scores)
    })

In [ ]:
for res in final_results:
    print("\n---------------------------")
    print("Model:", res["model"])
    print("Semantic Similarity:", round(res["similarity"], 4))
    print("BLEU:", round(res["bleu"], 4))
    print("ROUGE:", round(res["rouge"], 4))

In [ ]:
!ls /content

In [ ]:
!zip -r project_outputs.zip /content/results

In [ ]:
from google.colab import files
files.download("project_outputs.zip")

In [ ]:
print(len(data))